# HW5 – Illinois Professional Licenses Dataset
**IS 445 – Data Visualization**  
Samuel Sokolovsky

Dataset: [licenses_fall2022.csv](https://raw.githubusercontent.com/UIUC-iSchool-DataViz/is445_data/main/licenses_fall2022.csv)

In [3]:
import pandas as pd
import altair as alt

alt.data_transformers.disable_max_rows()

url = 'https://raw.githubusercontent.com/UIUC-iSchool-DataViz/is445_data/main/licenses_fall2022.csv'
df = pd.read_csv(url)
print(df.shape)
df.head()

(10000, 31)


,_id,License Type,Description,License Number,License Status,Business,Title,First Name,Middle,Last Name,...,Specialty/Qualifier,Controlled Substance Schedule,Delegated Controlled Substance Schedule,Ever Disciplined,LastModifiedDate,Case Number,Action,Discipline Start Date,Discipline End Date,Discipline Reason
0,1189509,DETECTIVE BOARD,PERMANENT EMPLOYEE REGISTRATION,129446286,NOT RENEWED,N,NaN,EILEEN,NaN,SANTACRUZ,...,None,None,None,N,03/18/2022,None,None,None,None,None
1,801037,DETECTIVE BOARD,FIREARM CONTROL CARD,229030294.0,NOT RENEWED,N,NaN,DAGMAR,J,NORDLUND,...,None,None,None,N,08/16/2006,None,None,None,None,None
2,365129,COSMO,LICENSED COSMETOLOGIST,11053076.0,NOT RENEWED,N,NaN,RADOJE,NaN,ZELENOVIC,...,None,None,None,N,05/26/2006,None,None,None,None,None
3,595427,COSMO,LICENSED COSMETOLOGIST,11295645.0,ACTIVE,N,NaN,BECKY SUE,L,BURROUGHS,...,None,None,None,N,11/12/2021,None,None,None,None,None
4,653668,COSMO,LICENSED NAIL TECHNICIAN,169006247,NOT RENEWED,N,NaN,BILL G,L,LETNER,...,None,None,None,N,05/30/2006,None,None,None,None,None


In [4]:
print(df.columns.tolist())
print(df['License Type'].value_counts().head(10))
print(df['License Status'].value_counts())
print(df['State'].value_counts().head(10))

['_id', 'License Type', 'Description', 'License Number', 'License Status', 'Business', 'Title', 'First Name', 'Middle', 'Last Name', 'Prefix', 'Suffix', 'Business Name', 'BusinessDBA', 'Original Issue Date', 'Effective Date', 'Expiration Date', 'City', 'State', 'Zip', 'County', 'Specialty/Qualifier', 'Controlled Substance Schedule', 'Delegated Controlled Substance Schedule', 'Ever Disciplined', 'LastModifiedDate', 'Case Number', 'Action', 'Discipline Start Date', 'Discipline End Date', 'Discipline Reason']
DETECTIVE BOARD           4867
COSMO                     3781
DENTAL                     739
FUNERAL AND EMBALMER        98
DIETETIC AND NUTRITION      73
DESIGN FIRM                 71
MASSAGE LICENSING BD        52
HOME INSPECTOR              46
COMM ASSOC MGR              37
CLIN PSYCHOLOGIST           24
Name: License Type, dtype: int64
NOT RENEWED                             6331
ACTIVE                                  2440
INACTIVE                                 532
TERMINATED

## Data Cleaning & Transformation

In [5]:
df['Original Issue Date'] = pd.to_datetime(df['Original Issue Date'], errors='coerce')
df['Expiration Date']     = pd.to_datetime(df['Expiration Date'],     errors='coerce')

df = df.dropna(subset=['Original Issue Date'])

df['Issue Year'] = df['Original Issue Date'].dt.year

df = df[(df['Issue Year'] >= 1970) & (df['Issue Year'] <= 2023)]

print(df.shape)

(9641, 32)


## Plot 1 – License Counts Over Time by License Status (Interactive)

This is our **interactive** chart. A dropdown/legend selection lets the viewer highlight one license status at a time to see how issuing rates changed across the years.

In [6]:
status_year = (
    df.groupby(['Issue Year', 'License Status'])
      .size()
      .reset_index(name='Count')
)

top_statuses = df['License Status'].value_counts().head(5).index.tolist()
status_year = status_year[status_year['License Status'].isin(top_statuses)]

selection = alt.selection_point(fields=['License Status'], bind='legend')

chart1 = (
    alt.Chart(status_year)
    .mark_line(point=True)
    .encode(
        x=alt.X('Issue Year:O',
                title='Year of Original Issue',
                axis=alt.Axis(labelAngle=-45)),
        y=alt.Y('Count:Q',
                title='Number of Licenses Issued'),
        color=alt.Color('License Status:N',
                        scale=alt.Scale(scheme='tableau10'),
                        title='License Status'),
        opacity=alt.condition(selection, alt.value(1.0), alt.value(0.1)),
        tooltip=['Issue Year:O', 'License Status:N', 'Count:Q']
    )
    .add_params(selection)
    .properties(
        width=650,
        height=350,
        title='Illinois Professional Licenses Issued Per Year by Status'
    )
)

chart1

alt.Chart(...)

In [7]:
chart1.save('chart1.html')

## Plot 2 – Top License Types by Count (Bar Chart)

A static horizontal bar chart showing the 15 most common license types in the dataset.

In [8]:
top_types = (
    df['License Type']
      .value_counts()
      .head(15)
      .reset_index()
)
top_types.columns = ['License Type', 'Count']

chart2 = (
    alt.Chart(top_types)
    .mark_bar()
    .encode(
        x=alt.X('Count:Q', title='Number of Licenses'),
        y=alt.Y('License Type:N',
                sort='-x',
                title='License Type'),
        color=alt.Color('Count:Q',
                        scale=alt.Scale(scheme='blues'),
                        title='Count'),
        tooltip=['License Type:N', 'Count:Q']
    )
    .properties(
        width=600,
        height=400,
        title='Top 15 Most Common Professional License Types in Illinois'
    )
)

chart2

alt.Chart(...)

In [9]:
chart2.save('chart2.html')